# OCO 제품별 edge / 안쪽 패턴 비교
기존 27·28번 저장 자료만 읽는 독립 노트북입니다. **00번 EP_BASE_DIR를 실제 BASIC 폴더 경로로 지정한 뒤 00→07 순서**로 실행하세요. 기본값은 `E:\PHOTO_ANALYSIS\BASIC`입니다. 커널의 임시 작업 폴더를 기준으로 삼지 않습니다.

제품마다 `27_OCO_FINAL_POINTS.parquet`와 같은 실행의 `extension_settings.json` 두 파일이 필요합니다. 전체 Excel도 지원하며 `_sample` 파일은 사용하지 않습니다. 기존 output 폴더 전체를 지정하면 stages 하위도 읽습니다. ADI 및 배율 sweep 파일은 불필요합니다.

배율 적용 전 CPE 입력(기존 decorrection·K-MRC 유지)을 복원합니다. 전체 wafer의 안쪽/edge 입력 통계와, 안쪽 point만으로 CPE38을 추정한 edge 예측 오차를 구분합니다. 예측 불안정/불가능 shot은 이유와 coverage를 기록합니다. **KGVM·KHVF는 Y17, PBVC(VC)는 X21**만 분석합니다. 제품별 방향은 00번 EP_SOURCES의 axis로 지정합니다. 그래프는 각 제품의 지정 방향을 함께 비교하며 축/범례에 X/Y를 명시합니다. DB 조회·기존 파일 수정은 하지 않습니다.


## 00. 제품별 입력 폴더·분석 설정


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import copy
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

EP_BASE_DIR = Path(r"E:\PHOTO_ANALYSIS\BASIC")  # 실제 BASIC 폴더 경로로 수정. 임시 실행 폴더 대신 이 경로를 사용
EP_SOURCES = [
    {"product": "KGVM", "axis": "y", "folder": EP_BASE_DIR / "edge_comparison_input" / "KGVM"},
    {"product": "KHVF", "axis": "y", "folder": EP_BASE_DIR / "edge_comparison_input" / "KHVF"},
    {"product": "PBVC", "axis": "x", "folder": EP_BASE_DIR / "edge_comparison_input" / "PBVC"},
]
EP_OUTPUT_DIR = EP_BASE_DIR / "output_edge_inner_comparison_critical"
EP_INNER_RADIUS_UM = 144000.0
EP_MAX_RADIUS_UM = 147000.0  # 실제 측정점 반경. 147mm 초과 및 반경 미상은 제외 수를 따로 기록
EP_STD_DDOF = 1
EP_SAVE_POINT_DATA = True  # edge가 있는 shot의 검산용 point 자료만 제품별 Parquet 저장
EP_MIN_EXTRA_INNER_POINTS = 2  # X: 최소 21+2, Y: 최소 17+2. para 수는 줄이지 않음
EP_MAX_SCALED_CONDITION = 1e6
EP_MAX_EDGE_NOISE_GAIN = 5.0  # 안쪽 회귀의 edge 예측 noise gain 진단용 상한. 공정 spec이 아님
EP_SVD_RCOND = 1e-12
EP_RADIAL_BINS_MM = np.arange(140., 147.01, .5)
EP_WAFER_KEYS = ["product", "LOT_ID", "Wafer"]
EP_SHOT_KEYS = EP_WAFER_KEYS + ["fcp_x", "fcp_y"]

if not EP_SOURCES or len({str(s["product"]) for s in EP_SOURCES}) != len(EP_SOURCES):
    raise ValueError("EP_SOURCES에 서로 다른 제품명과 입력 폴더를 지정하세요.")
if any(str(source.get("axis", "")).lower() not in {"x", "y"} for source in EP_SOURCES):
    raise ValueError("EP_SOURCES의 각 제품에 axis를 x 또는 y로 지정하세요.")
EP_PRODUCT_AXES = {str(source["product"]): [str(source["axis"]).lower()] for source in EP_SOURCES}
EP_AXES = sorted({axis for axes in EP_PRODUCT_AXES.values() for axis in axes})  # 내부 함수용 전체 방향 목록
if EP_STD_DDOF not in [0, 1] or not 0 < EP_INNER_RADIUS_UM < EP_MAX_RADIUS_UM:
    raise ValueError("표준편차 ddof 또는 반경 설정을 확인하세요.")
if EP_MIN_EXTRA_INNER_POINTS < 0 or not 0 < EP_SVD_RCOND < 1 or EP_MAX_EDGE_NOISE_GAIN <= 0 or EP_MAX_SCALED_CONDITION <= 1:
    raise ValueError("안쪽 기준 모델의 진단 설정을 확인하세요.")
for folder in [EP_OUTPUT_DIR, EP_OUTPUT_DIR / "tables", EP_OUTPUT_DIR / "points", EP_OUTPUT_DIR / "plots"]:
    folder.mkdir(parents=True, exist_ok=True)


def ep_locate_inputs(source):
    folder = Path(source["folder"])
    candidates = [folder / "27_OCO_FINAL_POINTS.parquet", folder / "stages" / "27_OCO_FINAL_POINTS.parquet",
                  folder / "27_OCO_FINAL_POINTS.xlsx", folder / "stages" / "27_OCO_FINAL_POINTS.xlsx"]
    existing = [p for p in candidates if p.is_file()]
    settings = folder / "extension_settings.json"
    if not existing or not settings.is_file():
        raise FileNotFoundError(f"{source['product']}: {folder}에 전체 27_OCO_FINAL_POINTS.parquet와 extension_settings.json을 두세요. stages 하위 파일 및 전체 Excel도 지원합니다. _sample 파일은 사용하지 않습니다.")
    return existing, settings


def ep_read_product(source):
    candidates, settings_path = ep_locate_inputs(source)
    errors = []
    for path in candidates:
        try:
            frame = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_excel(path)
            settings = json.loads(settings_path.read_text(encoding="utf-8"))
            return frame, settings, path, settings_path
        except (OSError, ValueError, ImportError) as exc:
            errors.append(f"{path.name}: {exc}")
    raise ValueError(f"{source['product']} 입력 읽기 실패: " + " | ".join(errors))


def ep_prepare_points(frame, product, axes=None):
    axes = EP_AXES if axes is None else axes
    required = ["LOT_ID", "Wafer", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y", "measurement_scale"] + [f"raw_minus_total_fit_{axis}" for axis in axes]
    missing = [col for col in required if col not in frame]
    if missing:
        raise ValueError(f"{product}: 27번 전체 point 자료에 필수 컬럼이 없습니다: {missing}")
    out = frame.copy().reset_index(drop=True)
    out["product"] = str(product)
    numeric = required[1:] + ["radius", "wf_x", "wf_y"] + [f"{axis.upper()}_reg_original" for axis in axes]
    for col in numeric:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if out.empty or out[EP_SHOT_KEYS].isna().any().any() or not np.isfinite(out[["Wafer", "fcp_x", "fcp_y"]].to_numpy(float)).all():
        raise ValueError(f"{product}: LOT/wafer/shot 식별자가 없거나 입력이 비어 있습니다.")
    if "photo_transn_seq" in out:
        photos = out.groupby(EP_WAFER_KEYS, observed=True, dropna=False).photo_transn_seq.nunique()
        if photos.gt(1).any():
            raise ValueError(f"{product}: 한 LOT/wafer에 여러 photo 이력이 있습니다. 실행별 자료를 섞지 말고 한 이력씩 분리하세요.")
    if "radius" not in out:
        out["radius"] = np.hypot(out.wf_x, out.wf_y) if {"wf_x", "wf_y"}.issubset(out.columns) else np.hypot(out.fcp_x + out.coordinate_X, out.fcp_y + out.coordinate_Y)
    elif out.radius.isna().any():
        fallback = np.hypot(out.wf_x, out.wf_y) if {"wf_x", "wf_y"}.issubset(out.columns) else np.hypot(out.fcp_x + out.coordinate_X, out.fcp_y + out.coordinate_Y)
        out["radius"] = out.radius.fillna(fallback)
    finite_scale = np.isfinite(out.measurement_scale)
    if not finite_scale.all() or out.measurement_scale.lt(0).any():
        raise ValueError(f"{product}: measurement_scale이 유한한 0 이상 값이어야 합니다.")
    scaled = out.measurement_scale.ne(1.)
    for axis in axes:
        col = f"{axis.upper()}_reg_original"
        needed = scaled & np.isfinite(out[f"raw_minus_total_fit_{axis}"])
        if needed.any() and (col not in out or not np.isfinite(out.loc[needed, col].to_numpy(float)).all()):
            raise ValueError(f"{product}: 배율 적용 전 값 복원에 필요한 {col}이 없습니다. 27번 전체 Parquet을 확인하세요.")
        restore = pd.Series(0., index=out.index)
        if needed.any():
            restore.loc[needed] = (1. - out.loc[needed, "measurement_scale"]) * out.loc[needed, col]
        # 기적용 MRC/PSM decorrection과 고정 K-MRC는 유지. 측정값 배율만 100%로 복원.
        out[f"input100_{axis}"] = out[f"raw_minus_total_fit_{axis}"] + restore
    in_range = np.isfinite(out.radius) & out.radius.ge(0) & out.radius.le(EP_MAX_RADIUS_UM)
    audit = dict(product=product, axis="/".join(axes), n_input=len(out), n_radius_unknown=int((~np.isfinite(out.radius)).sum()),
                 n_outside_radius=int((np.isfinite(out.radius) & ~in_range).sum()), n_scaled=int(scaled.sum()))
    out = out.loc[in_range].copy().reset_index(drop=True)
    if out.empty:
        raise ValueError(f"{product}: 분석 반경 안의 측정점이 없습니다.")
    out["region"] = np.where(out.radius.le(EP_INNER_RADIUS_UM), "INNER", "EDGE")
    out["_fit_valid"] = np.isfinite(out[["coordinate_X", "coordinate_Y"] + [f"input100_{axis}" for axis in axes]].to_numpy(float)).all(axis=1)
    out["_has_edge_shot"] = out.region.eq("EDGE").groupby([out[k] for k in EP_SHOT_KEYS], observed=True, dropna=False).transform("any")
    audit.update(n_used=len(out), n_inner=int(out.region.eq("INNER").sum()), n_edge=int(out.region.eq("EDGE").sum()), n_fit_valid=int(out._fit_valid.sum()))
    return out, audit


ep_input_inventory = []
for source in EP_SOURCES:
    paths, settings_path = ep_locate_inputs(source)
    ep_input_inventory.append(dict(product=source["product"], axis=EP_PRODUCT_AXES[str(source["product"])][0], point_files=" | ".join(str(p) for p in paths), settings_file=str(settings_path)))
print("분석 기준 폴더:", EP_BASE_DIR)
display(pd.DataFrame(ep_input_inventory))
print("00~07번을 순서대로 실행하세요. DB 조회 없이 제품별로 읽고 처리하며, 기존 입력 파일은 변경하지 않습니다.")


## 01. 통계·저장된 제품별 설정 복원 함수


In [ ]:
EP_STAT_COLUMNS = ["n_points", "n_valid", "mean_nm", "abs_mean_nm", "sigma_nm", "three_sigma_nm", "M3S_nm", "RMSE_nm", "p95_abs_nm"]


def ep_group_stats(frame, keys, value_col, skeleton=None):
    result = frame[keys].drop_duplicates().copy() if skeleton is None else skeleton[keys].drop_duplicates().copy()
    if frame.empty:
        for col in EP_STAT_COLUMNS:
            result[col] = 0 if col in ["n_points", "n_valid"] else np.nan
        return result
    work = frame[keys].copy()
    values = pd.to_numeric(frame[value_col], errors="coerce") * 1000
    work["_v"] = values.where(np.isfinite(values))
    work["_v2"], work["_abs"] = work._v ** 2, work._v.abs()
    groups = work.groupby(keys, observed=True, dropna=False)
    stats = groups._v.agg(n_points="size", n_valid="count", mean_nm="mean")
    stats["sigma_nm"] = groups._v.std(ddof=EP_STD_DDOF)
    stats["RMSE_nm"] = np.sqrt(groups._v2.mean())
    stats["p95_abs_nm"] = groups._abs.quantile(.95)
    result = result.merge(stats.reset_index(), on=keys, how="left", validate="1:1")
    result[["n_points", "n_valid"]] = result[["n_points", "n_valid"]].fillna(0).astype(int)
    result["abs_mean_nm"] = result.mean_nm.abs()
    result["three_sigma_nm"] = 3 * result.sigma_nm
    result["M3S_nm"] = result.abs_mean_nm + result.three_sigma_nm
    return result


def ep_input_stats(points, keys, axes=None):
    axes = EP_AXES if axes is None else axes
    tables = []
    for axis in axes:
        for region in ["INNER", "EDGE"]:
            table = ep_group_stats(points.loc[points.region.eq(region)], keys, f"input100_{axis}", skeleton=points)
            table["axis"], table["region"] = axis, region
            tables.append(table)
    return pd.concat(tables, ignore_index=True)


def ep_region_comparison(stats, keys):
    columns = keys + ["axis"]
    inner = stats.loc[stats.region.eq("INNER")].drop(columns="region")
    edge = stats.loc[stats.region.eq("EDGE")].drop(columns="region")
    result = inner.merge(edge, on=columns, suffixes=("_inner", "_edge"), validate="1:1")
    result["delta_M3S_edge_minus_inner_nm"] = result.M3S_nm_edge - result.M3S_nm_inner
    result["ratio_3sigma_edge_to_inner"] = result.three_sigma_nm_edge / result.three_sigma_nm_inner.replace(0, np.nan)
    return result


def ep_restore_settings(settings):
    required = ["cpe_options", "use_virtual_chip_points", "virtual_wafer_radius_um", "span_tolerance", "minimum_ridge", "add_ridge", "max_gopt_return", "max_span_iter", "cd_tol", "ridge_valves", "active_span_limits"]
    missing = [key for key in required if key not in settings]
    if missing:
        raise ValueError(f"28번 실행 설정에 필수 항목이 없습니다: {missing}")
    if settings.get("kmrc_definition") != "OCO_E1_RAW_MINUS_ADI_RAW":
        raise ValueError("현재 K-MRC(E1 raw−ADI raw) 방식의 27·28번 저장 자료가 필요합니다. 이전 E2 자체 regression 자료와 섞을 수 없습니다.")
    options = copy.deepcopy(settings["cpe_options"])
    globals_map = {"use_virtual_chip_points": "EP_USE_VIRTUAL_CHIP_POINTS", "virtual_wafer_radius_um": "EP_VIRTUAL_WAFER_RADIUS", "span_tolerance": "EP_SPAN_TOLERANCE",
                   "minimum_ridge": "EP_MINIMUM_RIDGE", "add_ridge": "EP_ADD_RIDGE", "max_gopt_return": "EP_MAX_GOPT_RETURN", "max_span_iter": "EP_MAX_SPAN_ITER", "cd_tol": "EP_CD_TOL"}
    for key, variable in globals_map.items():
        globals()[variable] = settings[key]
    globals()["EP_RIDGE_VALVE_HYPER"] = copy.deepcopy(settings["ridge_valves"])
    span_variables = {"OVO3_38P_HARDLIMIT": "EP_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT", "OVO3_38P_F2F": "EP_RK_SPAN_LIMITS_OVO3_38P_F2F"}
    if options["span_mode"] not in span_variables:
        raise ValueError(f"지원하지 않는 span_mode: {options['span_mode']}")
    globals()[span_variables[options["span_mode"]]] = copy.deepcopy(settings["active_span_limits"])
    return options


## 02. 기존 CPE38 모델·virtual grid


In [ ]:
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

EP_SPAN_MODE = "OVO3_38P_F2F"
EP_RIDGE_REGRESSION = True
EP_G_OPT_SPEC = 1.0
EP_USE_VIRTUAL_CHIP_POINTS = True   # VBA virtual grid: chip별 4 corner + center point
EP_G_OPT_TOLERANCE = 0.001          # VBA: G_opt_tolerlance = 0.001
EP_SPAN_TOLERANCE = 0.01            # VBA: span_tolerlance = 0.01
EP_MINIMUM_RIDGE = 0.001            # VBA: minimum_Ridge = 0.001
EP_ADD_RIDGE = 10.0                 # VBA: add_ridge = 10
EP_MAX_GOPT_RETURN = 50             # VBA: max_prevent_fallback_retrun = 50
EP_MAX_SPAN_ITER = 5000             # VBA: max_iter = 5000
EP_CD_TOL = 1e-6                    # VBA: tol = 0.000001
EP_CPE_EVAL_GRID_N = 10
EP_MIN_POINTS_GUIDE = 10
EP_G_OPTIMAL_TYPE = 1               # VBA information 기준: GOT1
EP_VIRTUAL_WAFER_RADIUS = 144000.0  # VBA virtual point keep 조건: R <= 144000               # VBA option D59: 1,2,3,4. 4 = inner OT2 / partial OT1
EP_USE_SPAN_BOX_CONSTRAINT = True   # VBA Balloon projected CD/PGD stage

def ep_cpe_38para(rx: Sequence[float], ry: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27,
        (rx ** 3 * ry ** 4) / 1e31,
    ]).T
    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23,
        (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27,
    ]).T
    return X_dx, X_dy
EP_CPE38_KEYS_DX = [
    "RK1", "RK3", "RK5",
    "RK7", "RK9", "RK11",
    "RK13", "RK15", "RK17", "RK19",
    "RK23", "RK25", "RK27", "RK29",
    "RK35", "RK37", "RK39", "RK41",
    "RK49", "RK51",
    "RK65",
]
EP_CPE38_KEYS_DY = [
    "RK2", "RK4", "RK6",
    "RK8", "RK10", "RK12",
    "RK14", "RK16", "RK18",
    "RK22", "RK24", "RK26",
    "RK32", "RK34", "RK36",
    "RK46", "RK48",
]
EP_RK_SPAN_LIMITS_OVO3_38P_F2F = {
    'RK1': {'min': -0.1, 'max': 0.1},
    'RK2': {'min': -0.1, 'max': 0.1},
    'RK3': {'min': -2.0, 'max': 2.0},
    'RK4': {'min': -10.0, 'max': 10.0},
    'RK5': {'min': -5.0, 'max': 5.0},
    'RK6': {'min': -5.0, 'max': 5.0},
    'RK7': {'min': -0.1, 'max': 0.1},
    'RK8': {'min': -0.15, 'max': 0.15},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -0.5, 'max': 0.5},
    'RK11': {'min': -0.5, 'max': 0.5},
    'RK12': {'min': -0.03, 'max': 0.03},
    'RK13': {'min': -0.003, 'max': 0.003},
    'RK14': {'min': -0.015, 'max': 0.015},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -0.05, 'max': 0.05},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -0.023, 'max': 0.023},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}
EP_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = {
    'RK1': {'min': -100.0, 'max': 100.0},
    'RK2': {'min': -100.0, 'max': 100.0},
    'RK3': {'min': -1000.0, 'max': 1000.0},
    'RK4': {'min': -1000.0, 'max': 1000.0},
    'RK5': {'min': -1000.0, 'max': 1000.0},
    'RK6': {'min': -1000.0, 'max': 1000.0},
    'RK7': {'min': -0.25, 'max': 0.25},
    'RK8': {'min': -10.0, 'max': 10.0},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -10.0, 'max': 10.0},
    'RK11': {'min': -10.0, 'max': 10.0},
    'RK12': {'min': -0.25, 'max': 0.25},
    'RK13': {'min': -0.025, 'max': 0.025},
    'RK14': {'min': -1.0, 'max': 1.0},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -1.0, 'max': 1.0},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -1.0, 'max': 1.0},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}

def ep_get_active_span_limits(span_mode: str = "OVO3_38P_F2F") -> Dict[str, Dict[str, float]]:
    if span_mode == "OVO3_38P_HARDLIMIT":
        return EP_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT
    return EP_RK_SPAN_LIMITS_OVO3_38P_F2F
EP_RIDGE_VALVE_HYPER = {
    "RK1": 0.0, "RK2": 0.0,
    "RK3": 0.01, "RK4": 0.01, "RK5": 0.01, "RK6": 0.01,
    "RK7": 1.0, "RK8": 0.1, "RK9": 1.0, "RK10": 0.1,
    "RK11": 0.1, "RK12": 1.0,
    "RK13": 1.0, "RK14": 0.1, "RK15": 1.0, "RK16": 0.1,
    "RK17": 1.0, "RK18": 1.0, "RK19": 0.1, "RK20": 1.0,
}

def ep__to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def ep_choose_base_lambda_scale(n_points: int) -> float:
    """VBA의 point가 적어도 skip하지 않는 방식을 Python식으로 반영."""
    if n_points >= 10:
        return 1.0
    if n_points >= 7:
        return 1e2
    if n_points >= 5:
        return 1e4
    if n_points >= 3:
        return 1e6
    return 1e8

def ep_get_ridge_valves(coeff_names: Sequence[str]) -> np.ndarray:
    return np.array([EP_RIDGE_VALVE_HYPER.get(name, 1.0) for name in coeff_names], dtype=float)

def ep_make_lambda_vector(base_lambda: float, coeff_names: Sequence[str]) -> np.ndarray:
    valves = ep_get_ridge_valves(coeff_names)
    return float(base_lambda) * valves

def ep_build_virtual_chip_points(
    step_pitch_x: float,
    step_pitch_y: float,
    chip_x_num: int,
    chip_y_num: int,
    round_decimals: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA virtual grid 재현:
    chip별 4 corner + center point를 shot-local coordinate로 생성하고,
    chip 경계에서 겹치는 corner는 dedup한다.
    예: chip_x_num=2, chip_y_num=1
      - 각 chip 5 point => 10 raw points
      - 가운데 공유 corner 2개 중복 제거
      - 최종 8 virtual points
    coordinate_X/Y가 shot center 기준이라고 보고,
    전체 shot 영역은 [-STEP_PITCH_X/2, +STEP_PITCH_X/2],
                  [-STEP_PITCH_Y/2, +STEP_PITCH_Y/2] 로 둔다.
    """
    chip_x_num = int(chip_x_num) if pd.notna(chip_x_num) and int(chip_x_num) > 0 else 1
    chip_y_num = int(chip_y_num) if pd.notna(chip_y_num) and int(chip_y_num) > 0 else 1
    sx = float(step_pitch_x)
    sy = float(step_pitch_y)
    x_edges = np.linspace(-sx / 2.0, sx / 2.0, chip_x_num + 1)
    y_edges = np.linspace(-sy / 2.0, sy / 2.0, chip_y_num + 1)
    pts = []
    for ix in range(chip_x_num):
        for iy in range(chip_y_num):
            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]
            xc = (x0 + x1) / 2.0
            yc = (y0 + y1) / 2.0
            pts.extend([
                (x0, y0), (x0, y1), (x1, y0), (x1, y1),
                (xc, yc),
            ])
    pts_unique = sorted({(round(x, round_decimals), round(y, round_decimals)) for x, y in pts})
    rx = np.array([p[0] for p in pts_unique], dtype=float)
    ry = np.array([p[1] for p in pts_unique], dtype=float)
    return rx, ry

def ep_filter_virtual_points_by_wafer_radius(
    rx_v: Sequence[float],
    ry_v: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = EP_VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    VBA edge-shot virtual point filtering.
    virtual point의 wafer 좌표 R이 wafer_radius 이하인 point만 사용한다.
    R = sqrt((fcp_x + rx_v)^2 + (fcp_y + ry_v)^2)
    keep if R <= 144000
    """
    rx_v = np.asarray(rx_v, dtype=float)
    ry_v = np.asarray(ry_v, dtype=float)
    wx = float(fcp_x) + rx_v
    wy = float(fcp_y) + ry_v
    r = np.sqrt(wx * wx + wy * wy)
    keep = r <= float(wafer_radius)
    return rx_v[keep], ry_v[keep], r[keep]

def ep_get_virtual_points_for_group(
    grp: pd.DataFrame,
    fallback_rx: Sequence[float],
    fallback_ry: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = EP_VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, str, int, int]:
    """
    1) full virtual point 생성: chip별 4 corner + center
    2) edge shot 처리: wafer 좌표 R <= wafer_radius point만 keep
    반환:
      rx_v, ry_v, virtual_source, n_virtual_full, n_virtual_kept
    """
    required = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
    if all(c in grp.columns for c in required):
        vals = grp[required].dropna()
        if len(vals) > 0:
            row = vals.iloc[0]
            rx_full, ry_full = ep_build_virtual_chip_points(
                step_pitch_x=row["STEP_PITCH_X"],
                step_pitch_y=row["STEP_PITCH_Y"],
                chip_x_num=row["CHIP_X_NUM"],
                chip_y_num=row["CHIP_Y_NUM"],
            )
            n_full = len(rx_full)
            rx_keep, ry_keep, _ = ep_filter_virtual_points_by_wafer_radius(
                rx_full,
                ry_full,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=wafer_radius,
            )
            if len(rx_keep) > 0:
                src = "chip_corner_center_wafer_clipped"
                return rx_keep, ry_keep, src, n_full, len(rx_keep)
            return rx_full, ry_full, "chip_corner_center_no_clip_points_fallback", n_full, n_full
    rx_v, ry_v = ep_build_eval_grid_from_data(fallback_rx, fallback_ry, grid_n=EP_CPE_EVAL_GRID_N)
    return rx_v, ry_v, "measured_range_grid_fallback", len(rx_v), len(rx_v)


## 03. 기존 ridge/G-opt/span solver


In [ ]:
def ep_stable_inverse_or_pinv(A: np.ndarray) -> np.ndarray:
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def ep_ridge_predict(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X.T @ X + np.diag(lambdas)
    b = X.T @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b
    return beta, X @ beta

def ep_build_eval_grid_from_data(rx: Sequence[float], ry: Sequence[float], grid_n: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    rx_min, rx_max = np.nanmin(rx), np.nanmax(rx)
    ry_min, ry_max = np.nanmin(ry), np.nanmax(ry)
    if not np.isfinite(rx_min) or not np.isfinite(rx_max):
        rx_min, rx_max = -1.0, 1.0
    if not np.isfinite(ry_min) or not np.isfinite(ry_max):
        ry_min, ry_max = -1.0, 1.0
    if rx_min == rx_max:
        rx_min -= 1.0
        rx_max += 1.0
    if ry_min == ry_max:
        ry_min -= 1.0
        ry_max += 1.0
    gx = np.linspace(rx_min, rx_max, grid_n)
    gy = np.linspace(ry_min, ry_max, grid_n)
    RX, RY = np.meshgrid(gx, gy)
    return RX.ravel(), RY.ravel()

def ep_gopt_value(
    X_meas: np.ndarray,
    X_eval: np.ndarray,
    lambdas: np.ndarray,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
) -> float:
    """
    VBA G_optimal_type 반영.
    - Type1: H H.T diag의 sqrt(max)
    - Type2: H matrix element의 signed max의 sqrt. 음수이면 999.0 반환.
    - Type3: X_eval inv(...) X_eval.T diag의 sqrt(max)
    - Type4: inner=Type2, partial=Type1
    """
    X_meas = np.asarray(X_meas, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X_meas.T @ X_meas + np.diag(lambdas)
    invA = ep_stable_inverse_or_pinv(A)
    effective_type = gopt_type
    if gopt_type == 4:
        effective_type = 2 if inner_or_partial == "inner" else 1
    if effective_type == 1:
        H = X_eval @ invA @ X_meas.T
        diag_val = np.sum(H * H, axis=1)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    if effective_type == 2:
        H = X_eval @ invA @ X_meas.T
        max_val = float(np.nanmax(H))
        if max_val < 0:
            return 999.0
        return float(np.sqrt(max_val))
    if effective_type == 3:
        M = X_eval @ invA @ X_eval.T
        diag_val = np.diag(M)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    raise ValueError(f"Unsupported gopt_type: {gopt_type}")

def ep_calc_contribution_spans(X_eval: np.ndarray, beta: np.ndarray, coeff_names: Sequence[str]) -> Dict[str, float]:
    spans = {}
    for i, name in enumerate(coeff_names):
        contrib = X_eval[:, i] * beta[i]
        spans[name] = float(np.nanmax(contrib) - np.nanmin(contrib))
    return spans

def ep_find_span_violations(
    spans: Dict[str, float],
    span_limits: Dict[str, Dict[str, float]],
    tolerance: float = 0.01,
) -> List[Tuple[str, float, float, float]]:
    violations = []
    for name, span in spans.items():
        if name not in span_limits:
            continue
        lo = span_limits[name]["min"] - tolerance
        hi = span_limits[name]["max"] + tolerance
        if span < lo or span > hi:
            violations.append((name, span, span_limits[name]["min"], span_limits[name]["max"]))
    return violations

def ep_build_beta_bounds_from_span(
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA의 min_Span_X/Y, max_Span_X/Y에 대응하는 box constraint 근사.
    contribution span = range(X_j * beta_j)가 span_limit 안에 들어오도록
    beta_j bound를 계산한다.
    """
    n = len(coeff_names)
    lo = np.full(n, -np.inf, dtype=float)
    hi = np.full(n, np.inf, dtype=float)
    for j, name in enumerate(coeff_names):
        if name not in span_limits:
            continue
        lim = span_limits[name]
        span_abs = max(abs(float(lim["min"])), abs(float(lim["max"])))
        xj_range = float(np.nanmax(X_eval[:, j]) - np.nanmin(X_eval[:, j]))
        if xj_range <= 0 or not np.isfinite(xj_range):
            continue
        b = span_abs / xj_range
        lo[j] = -b
        hi[j] = b
    return lo, hi

def ep_projected_coordinate_descent_ridge(
    X: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    beta_init: Optional[np.ndarray] = None,
    lower: Optional[np.ndarray] = None,
    upper: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 5000,
) -> Tuple[np.ndarray, np.ndarray, int, float]:
    """
    VBA ProjectedCoordinateDescentElasticNet(l1=0) 근사.
    solve min ||y-Xb||^2 + sum(lambda_j*b_j^2), s.t. lower <= b <= upper
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    n_param = X.shape[1]
    if beta_init is None:
        beta, _ = ep_ridge_predict(X, y, lambdas)
    else:
        beta = np.asarray(beta_init, dtype=float).copy()
    if lower is None:
        lower = np.full(n_param, -np.inf)
    if upper is None:
        upper = np.full(n_param, np.inf)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    beta = np.clip(beta, lower, upper)
    residual = y - X @ beta
    col_norm2 = np.sum(X * X, axis=0)
    last_delta = np.inf
    for it in range(1, max_iter + 1):
        max_delta = 0.0
        for j in range(n_param):
            denom = col_norm2[j] + lambdas[j]
            if denom <= 0 or not np.isfinite(denom):
                continue
            residual += X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            new_beta = rho / denom
            new_beta = min(max(new_beta, lower[j]), upper[j])
            delta = abs(new_beta - beta[j])
            beta[j] = new_beta
            residual -= X[:, j] * beta[j]
            if delta > max_delta:
                max_delta = delta
        last_delta = max_delta
        if max_delta < tol:
            break
    return beta, X @ beta, it, float(last_delta)

@dataclass
class EP_RidgeFitResult:
    beta: np.ndarray
    pred: np.ndarray
    lambdas: np.ndarray
    gopt: float
    spans: Dict[str, float]
    violations: List[Tuple[str, float, float, float]]
    gopt_iter: int
    span_iter: int
    method: str

def ep_solve_vba_style_ridge_one_axis(
    X_fit: np.ndarray,
    y: np.ndarray,
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
    ridge_regression: bool = True,
    gopt_spec: float = 1.0,
    gopt_tolerance: float = 0.001,
    minimum_ridge: float = 0.001,
    add_ridge: float = 10.0,
    max_gopt_return: int = 50,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
    use_span_box_constraint: bool = True,
    span_tolerance: float = 0.01,
    cd_tol: float = 1e-6,
    max_span_iter: int = 5000,
    base_lambda_scale: float = 1.0,
    debug: bool = False,
) -> EP_RidgeFitResult:
    """
    VBA flow에 맞춘 한 방향(X or Y) fitting.
    - 초기 ridge는 0에 가까운 값이 아니라 VBA minimum_Ridge 기준 사용
    - Gopt가 spec 이상이면 ridge를 빠르게 키운 뒤 이분탐색으로 spec 근처를 찾음
    - span over가 남으면 projected coordinate descent로 coefficient를 box constraint 안에 넣음
    """
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    y = np.asarray(y, dtype=float)
    n_param = X_fit.shape[1]
    if n_param != len(coeff_names):
        raise ValueError(f"n_param({n_param}) != len(coeff_names)({len(coeff_names)})")
    initial_lambda = 0.0 if not ridge_regression else 0.0
    lambdas = ep_make_lambda_vector(initial_lambda, coeff_names)
    beta, pred = ep_ridge_predict(X_fit, y, lambdas)
    g = ep_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
    gopt_iter = 0
    method = "ols"
    if ridge_regression and (not np.isfinite(g) or g >= gopt_spec or g == 0):
        low_lambda = 0.0
        high_lambda = max(minimum_ridge * base_lambda_scale, minimum_ridge)
        high_lambda = max(high_lambda, minimum_ridge)
        high_lambdas = ep_make_lambda_vector(high_lambda, coeff_names)
        high_g = ep_gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        while (not np.isfinite(high_g) or high_g >= gopt_spec) and gopt_iter < max_gopt_return:
            low_lambda = high_lambda
            high_lambda = high_lambda * add_ridge
            high_lambdas = ep_make_lambda_vector(high_lambda, coeff_names)
            high_g = ep_gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
            gopt_iter += 1
            if debug:
                print(f"fast ridge: lambda={high_lambda:.6g}, gopt={high_g:.6g}")
        target_low = max(0.0, gopt_spec - gopt_tolerance)
        target_high = gopt_spec
        best_lambda = high_lambda
        best_g = high_g
        if np.isfinite(high_g) and high_g < gopt_spec:
            lo = low_lambda
            hi = high_lambda
            for _ in range(max_gopt_return):
                mid = (lo + hi) / 2.0
                mid_lambdas = ep_make_lambda_vector(mid, coeff_names)
                mid_g = ep_gopt_value(X_fit, X_eval, mid_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
                gopt_iter += 1
                if debug:
                    print(f"slow ridge: lambda={mid:.6g}, gopt={mid_g:.6g}")
                if target_low <= mid_g < target_high:
                    best_lambda = mid
                    best_g = mid_g
                    break
                if mid_g >= gopt_spec or not np.isfinite(mid_g):
                    lo = mid
                else:
                    hi = mid
                    best_lambda = mid
                    best_g = mid_g
                if abs(hi - lo) < max(minimum_ridge * 1e-3, 1e-12):
                    break
        lambdas = ep_make_lambda_vector(best_lambda, coeff_names)
        beta, pred = ep_ridge_predict(X_fit, y, lambdas)
        g = ep_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt"
    spans = ep_calc_contribution_spans(X_eval, beta, coeff_names)
    violations = ep_find_span_violations(spans, span_limits, tolerance=span_tolerance)
    span_iter = 0
    if ridge_regression and use_span_box_constraint and len(violations) > 0:
        lower, upper = ep_build_beta_bounds_from_span(X_eval, coeff_names, span_limits)
        beta_cd, pred_cd, span_iter, _ = ep_projected_coordinate_descent_ridge(
            X_fit,
            y,
            lambdas,
            beta_init=beta,
            lower=lower,
            upper=upper,
            tol=cd_tol,
            max_iter=max_span_iter,
        )
        beta = beta_cd
        pred = pred_cd
        spans = ep_calc_contribution_spans(X_eval, beta, coeff_names)
        violations = ep_find_span_violations(spans, span_limits, tolerance=span_tolerance)
        g = ep_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt_box_cd"
    return EP_RidgeFitResult(
        beta=beta,
        pred=pred,
        lambdas=lambdas,
        gopt=float(g),
        spans=spans,
        violations=violations,
        gopt_iter=int(gopt_iter),
        span_iter=int(span_iter),
        method=method,
    )

def ep_determine_inner_or_partial(n_valid: int, n_virtual: int) -> str:
    """VBA: virtual_points = Vgrid_XY면 inner, 아니면 partial."""
    return "inner" if n_valid >= n_virtual else "partial"


## 04. 안쪽-only CPE38 참조 및 edge 예측


In [ ]:
def ep_geometry(X, X_edge):
    norms = np.linalg.norm(X, axis=0)
    safe = np.where(norms > 0, norms, 1.)
    scaled = X / safe
    singular = np.linalg.svd(scaled, compute_uv=False)
    cutoff = singular[0] * EP_SVD_RCOND if len(singular) else 0.
    rank = int((singular > cutoff).sum())
    full_rank = rank == X.shape[1]
    condition = float(singular[0] / singular[-1]) if full_rank else np.inf
    transfer = (X_edge / safe) @ np.linalg.pinv(scaled, rcond=EP_SVD_RCOND)
    gains = np.linalg.norm(transfer, axis=1)
    return dict(rank=rank, scaled_condition=condition, ols_edge_noise_gain_max=float(gains.max()), ols_edge_noise_gain_rms=float(np.sqrt(np.mean(gains**2))))


def ep_ridge_edge_gain(X, X_edge, lambdas):
    # Box 제약 활성화 전 선형 ridge의 noise gain. CD 제약이 작동하면 근사 진단값이다.
    norms = np.linalg.norm(X, axis=0)
    safe = np.where(norms > 0, norms, 1.)
    augmented = np.vstack([X / safe, np.diag(np.sqrt(np.maximum(lambdas, 0.)) / safe)])
    target = np.vstack([np.eye(len(X)), np.zeros((X.shape[1], len(X)))])
    transfer = (X_edge / safe) @ np.linalg.lstsq(augmented, target, rcond=EP_SVD_RCOND)[0]
    gains = np.linalg.norm(transfer, axis=1)
    return float(gains.max()), float(np.sqrt(np.mean(gains**2)))


def ep_fit_inner_reference(points, options, axes=None):
    axes = EP_AXES if axes is None else axes
    out = points.copy()
    for axis in axes:
        out[f"inner_prediction_{axis}"], out[f"reference_residual_{axis}"], out[f"edge_deviation_{axis}"] = np.nan, np.nan, np.nan
        out[f"reference_eligible_{axis}"] = False
        out[f"reference_status_{axis}"] = "no_edge"
    rows, coefficients = [], []
    spans = ep_get_active_span_limits(options["span_mode"])
    for key, shot in out.loc[out._has_edge_shot].groupby(EP_SHOT_KEYS, observed=True, dropna=False):
        valid = shot.loc[shot._fit_valid]
        inner, edge = valid.loc[valid.region.eq("INNER")], valid.loc[valid.region.eq("EDGE")]
        identity = dict(zip(EP_SHOT_KEYS, key))
        identity.update({col: shot[col].iloc[0] if col in shot else np.nan for col in ["DieX", "DieY", "photo_transn_seq"]})
        for axis_index, axis in enumerate(["x", "y"]):
            if axis not in axes:
                continue
            names = EP_CPE38_KEYS_DX if axis == "x" else EP_CPE38_KEYS_DY
            row = dict(identity, axis=axis, n_inner=len(inner), n_edge=len(edge), n_parameters=len(names), eligible=False, status="not_evaluated",
                       rank=np.nan, scaled_condition=np.nan, ols_edge_noise_gain_max=np.nan, ridge_edge_noise_gain_max=np.nan, ridge_edge_noise_gain_rms=np.nan,
                       method=None, gopt=np.nan, lambda_max=np.nan, span_violation_count=np.nan, span_iter=np.nan, inner_RMSE_nm=np.nan, candidate_edge_RMSE_nm=np.nan)
            reasons = []
            if not len(edge):
                reasons.append("no_valid_edge_points")
            if len(inner) < len(names) + EP_MIN_EXTRA_INNER_POINTS:
                reasons.append("too_few_inner_points")
            if reasons:
                row["status"] = "|".join(reasons)
                out.loc[shot.index, f"reference_status_{axis}"] = row["status"]
                rows.append(row)
                continue
            X = ep_cpe_38para(inner.coordinate_X, inner.coordinate_Y)[axis_index]
            Xe = ep_cpe_38para(edge.coordinate_X, edge.coordinate_Y)[axis_index]
            geometry = ep_geometry(X, Xe)
            row.update(geometry)
            if geometry["rank"] < len(names):
                row["status"] = "inner_rank_deficient"
                out.loc[shot.index, f"reference_status_{axis}"] = row["status"]
                rows.append(row)
                continue
            if geometry["scaled_condition"] > EP_MAX_SCALED_CONDITION:
                reasons.append("inner_condition_high")
            if EP_USE_VIRTUAL_CHIP_POINTS:
                vx, vy, virtual_source, n_full, n_virtual = ep_get_virtual_points_for_group(valid, valid.coordinate_X, valid.coordinate_Y, fcp_x=key[-2], fcp_y=key[-1], wafer_radius=EP_VIRTUAL_WAFER_RADIUS)
            else:
                vx, vy = ep_build_eval_grid_from_data(valid.coordinate_X, valid.coordinate_Y, grid_n=options["eval_grid_n"])
                virtual_source, n_full, n_virtual = "measured_range_grid", len(vx), len(vx)
            Xv = ep_cpe_38para(vx, vy)[axis_index]
            try:
                fit = ep_solve_vba_style_ridge_one_axis(X_fit=X, y=inner[f"input100_{axis}"].to_numpy(float), X_eval=Xv, coeff_names=names, span_limits=spans,
                    ridge_regression=options["ridge_regression"], gopt_spec=options["gopt_spec"], gopt_tolerance=options["gopt_tolerance"], minimum_ridge=EP_MINIMUM_RIDGE,
                    add_ridge=EP_ADD_RIDGE, max_gopt_return=EP_MAX_GOPT_RETURN, gopt_type=options["gopt_type"], inner_or_partial=ep_determine_inner_or_partial(len(inner), n_virtual),
                    use_span_box_constraint=options["use_span_box_constraint"], span_tolerance=EP_SPAN_TOLERANCE, cd_tol=EP_CD_TOL, max_span_iter=EP_MAX_SPAN_ITER,
                    base_lambda_scale=ep_choose_base_lambda_scale(len(inner)))
                gain_max, gain_rms = ep_ridge_edge_gain(X, Xe, fit.lambdas)
                Xall = ep_cpe_38para(valid.coordinate_X, valid.coordinate_Y)[axis_index]
                prediction = Xall @ fit.beta
                errors = valid[f"input100_{axis}"].to_numpy(float) - prediction
                edge_error = errors[valid.region.eq("EDGE").to_numpy()]
                if not np.isfinite(prediction).all() or not np.isfinite(gain_max):
                    reasons.append("nonfinite_prediction")
                if gain_max > EP_MAX_EDGE_NOISE_GAIN:
                    reasons.append("edge_prediction_noise_high")
                if fit.violations:
                    reasons.append("remaining_span_violations")
                if options["ridge_regression"] and (not np.isfinite(fit.gopt) or fit.gopt > options["gopt_spec"] + options["gopt_tolerance"]):
                    reasons.append("gopt_not_met")
                row.update(ridge_edge_noise_gain_max=gain_max, ridge_edge_noise_gain_rms=gain_rms, method=fit.method, gopt=fit.gopt, lambda_max=float(np.max(fit.lambdas)),
                           span_violation_count=len(fit.violations), span_iter=fit.span_iter, virtual_source=virtual_source, n_virtual=n_virtual, n_virtual_full=n_full,
                           inner_RMSE_nm=float(np.sqrt(np.mean((inner[f"input100_{axis}"].to_numpy(float)-fit.pred)**2))*1000), candidate_edge_RMSE_nm=float(np.sqrt(np.mean(edge_error**2))*1000))
                row["eligible"], row["status"] = not reasons, "|".join(reasons) if reasons else "ok"
                out.loc[valid.index, f"inner_prediction_{axis}"] = prediction
                out.loc[valid.index, f"reference_residual_{axis}"] = errors
                out.loc[shot.index, f"reference_eligible_{axis}"] = row["eligible"]
                if row["eligible"]:
                    out.loc[edge.index, f"edge_deviation_{axis}"] = edge_error
                coefficient = dict(identity, axis=axis)
                for name, beta, ridge in zip(names, fit.beta, fit.lambdas):
                    coefficient[name], coefficient[f"lambda_{name}"] = float(beta), float(ridge)
                coefficients.append(coefficient)
            except (np.linalg.LinAlgError, ValueError, FloatingPointError) as exc:
                row["status"], row["error_message"] = "solver_error", str(exc)
            out.loc[shot.index, f"reference_status_{axis}"] = row["status"]
            rows.append(row)
    columns = EP_SHOT_KEYS + ["axis", "n_inner", "n_edge", "n_parameters", "eligible", "status", "rank", "scaled_condition", "ridge_edge_noise_gain_max", "candidate_edge_RMSE_nm"]
    diagnostics = pd.DataFrame(rows) if rows else pd.DataFrame(columns=columns)
    coeff = pd.DataFrame(coefficients) if coefficients else pd.DataFrame(columns=EP_SHOT_KEYS + ["axis"])
    return out, diagnostics, coeff


## 05. 제품별 실행·wafer/shot 비교표


In [ ]:
def ep_shot_metadata(points):
    columns = [col for col in ["DieX", "DieY", "photo_transn_seq"] if col in points]
    return points.groupby(EP_SHOT_KEYS, observed=True, dropna=False)[columns].first().reset_index() if columns else points[EP_SHOT_KEYS].drop_duplicates()


def ep_prediction_stats(points, keys, axes=None):
    axes = EP_AXES if axes is None else axes
    tables = []
    edge = points.loc[points.region.eq("EDGE")]
    for axis in axes:
        table = ep_group_stats(edge, keys, f"edge_deviation_{axis}", skeleton=points)
        candidate = edge.groupby(keys, observed=True, dropna=False).size().rename("n_edge_total").reset_index()
        table = table.merge(candidate, on=keys, how="left", validate="1:1")
        table["n_edge_total"] = table.n_edge_total.fillna(0).astype(int)
        table["coverage_fraction"] = table.n_valid / table.n_edge_total.replace(0, np.nan)
        table["axis"] = axis
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


def ep_radial_profile(points, axes=None):
    axes = EP_AXES if axes is None else axes
    selected = points.loc[points._has_edge_shot].copy()
    bin_index = np.searchsorted(EP_RADIAL_BINS_MM, selected.radius.to_numpy(float)/1000, side="right") - 1
    bin_index[selected.radius.eq(EP_RADIAL_BINS_MM[-1]*1000)] = len(EP_RADIAL_BINS_MM)-2
    selected["radial_bin"] = bin_index
    selected = selected.loc[selected.radial_bin.ge(0) & selected.radial_bin.lt(len(EP_RADIAL_BINS_MM)-1)]
    tables = []
    keys = EP_WAFER_KEYS + ["radial_bin"]
    for axis in axes:
        part = selected.loc[selected[f"reference_eligible_{axis}"]]
        table = ep_group_stats(part, keys, f"reference_residual_{axis}")
        table["axis"] = axis
        if not table.empty:
            index = table.radial_bin.astype(int).to_numpy()
            table["radius_mid_mm"] = (EP_RADIAL_BINS_MM[index]+EP_RADIAL_BINS_MM[index+1])/2
        else:
            table["radius_mid_mm"] = pd.Series(dtype=float)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


ep_wafer_input, ep_shot_input, ep_wafer_prediction, ep_shot_prediction = [], [], [], []
ep_reference_diagnostics, ep_reference_coefficients, ep_profiles, ep_audits, ep_run_sources = [], [], [], [], []
for source_index, source in enumerate(EP_SOURCES):
    axes = EP_PRODUCT_AXES[str(source["product"])]
    raw, settings, point_path, settings_path = ep_read_product(source)
    options = ep_restore_settings(settings)  # 제품별 저장 설정을 매번 복원
    points, audit = ep_prepare_points(raw, source["product"], axes=axes)
    del raw
    before_wafer = ep_input_stats(points, EP_WAFER_KEYS, axes=axes)
    before_shot = ep_input_stats(points.loc[points._has_edge_shot], EP_SHOT_KEYS, axes=axes)
    result, diagnostic, coefficients = ep_fit_inner_reference(points, options, axes=axes)
    metadata = ep_shot_metadata(result)
    before_shot = before_shot.merge(metadata, on=EP_SHOT_KEYS, how="left", validate="m:1")
    wafer_prediction = ep_prediction_stats(result, EP_WAFER_KEYS, axes=axes)
    shot_prediction = ep_prediction_stats(result.loc[result._has_edge_shot], EP_SHOT_KEYS, axes=axes)
    shot_prediction = shot_prediction.merge(metadata, on=EP_SHOT_KEYS, how="left", validate="m:1")
    profile = ep_radial_profile(result, axes=axes)
    ep_wafer_input.append(before_wafer)
    ep_shot_input.append(before_shot)
    ep_wafer_prediction.append(wafer_prediction)
    ep_shot_prediction.append(shot_prediction)
    ep_reference_diagnostics.append(diagnostic)
    ep_reference_coefficients.append(coefficients)
    ep_profiles.append(profile)
    ep_audits.append(audit)
    ep_run_sources.append(dict(product=source["product"], axis=axes[0], points=str(point_path), settings=str(settings_path), original_settings=settings))
    if EP_SAVE_POINT_DATA:
        columns = EP_SHOT_KEYS + [col for col in ["DieX", "DieY", "photo_transn_seq", "TEST", "GROUP", "coordinate_X", "coordinate_Y", "radius", "region", "measurement_source", "measurement_scale", "_fit_valid"] if col in result]
        columns += [col for axis in axes for col in [f"{axis.upper()}_reg_original", f"raw_minus_total_fit_{axis}", f"input100_{axis}"] if col in result]
        columns += [f"{stem}_{axis}" for axis in axes for stem in ["inner_prediction", "reference_residual", "edge_deviation", "reference_eligible", "reference_status"]]
        result["analysis_axis"] = axes[0]
        columns += ["analysis_axis"]
        result.loc[result._has_edge_shot, columns].to_parquet(EP_OUTPUT_DIR / "points" / f"Product_{source_index+1:02d}_edge_shots.parquet", index=False)
    print(f"{source['product']} [{axes[0].upper()}]: wafer {len(points[EP_WAFER_KEYS].drop_duplicates()):,}, inner {audit['n_inner']:,} pts, edge {audit['n_edge']:,} pts, reference eligible {int(diagnostic.eligible.sum()):,}/{len(diagnostic):,} shots")
    del points, result

ep_wafer_input = pd.concat(ep_wafer_input, ignore_index=True)
ep_shot_input = pd.concat(ep_shot_input, ignore_index=True)
ep_wafer_prediction = pd.concat(ep_wafer_prediction, ignore_index=True)
ep_shot_prediction = pd.concat(ep_shot_prediction, ignore_index=True)
ep_reference_diagnostics = pd.concat(ep_reference_diagnostics, ignore_index=True)
ep_reference_coefficients = pd.concat(ep_reference_coefficients, ignore_index=True)
ep_radial_profiles = pd.concat(ep_profiles, ignore_index=True)
ep_input_audit = pd.DataFrame(ep_audits)
ep_wafer_comparison = ep_region_comparison(ep_wafer_input, EP_WAFER_KEYS)
ep_shot_comparison = ep_region_comparison(ep_shot_input.drop(columns=["DieX", "DieY", "photo_transn_seq"], errors="ignore"), EP_SHOT_KEYS)
ep_shot_comparison = ep_shot_comparison.loc[ep_shot_comparison.n_valid_inner.gt(0) & ep_shot_comparison.n_valid_edge.gt(0)].copy()
ep_shot_comparison = ep_shot_comparison.merge(ep_shot_input[EP_SHOT_KEYS + [col for col in ["DieX", "DieY", "photo_transn_seq"] if col in ep_shot_input]].drop_duplicates(EP_SHOT_KEYS), on=EP_SHOT_KEYS, how="left", validate="m:1")
ep_prediction_coverage = ep_reference_diagnostics.groupby(["product", "axis", "status"], observed=True, dropna=False).size().rename("n_shots").reset_index()

ep_summary_rows = []
for product in [str(s["product"]) for s in EP_SOURCES]:
    for axis in EP_PRODUCT_AXES[product]:
        base = ep_wafer_comparison.loc[ep_wafer_comparison["product"].eq(product) & ep_wafer_comparison.axis.eq(axis)]
        prediction = ep_wafer_prediction.loc[ep_wafer_prediction["product"].eq(product) & ep_wafer_prediction.axis.eq(axis)]
        diag = ep_reference_diagnostics.loc[ep_reference_diagnostics["product"].eq(product) & ep_reference_diagnostics.axis.eq(axis)]
        ep_summary_rows.append(dict(product=product, axis=axis, n_wafers=len(base), n_wafers_with_prediction=int(prediction.n_valid.gt(0).sum()), n_edge_shots=len(diag), n_eligible_shots=int(diag.eligible.sum()),
            n_edge_total=int(prediction.n_edge_total.sum()), n_edge_predicted=int(prediction.n_valid.sum()), edge_prediction_coverage=float(prediction.n_valid.sum()/prediction.n_edge_total.sum()) if prediction.n_edge_total.sum() else np.nan,
            median_wafer_inner_M3S_nm=base.M3S_nm_inner.median(), median_wafer_edge_M3S_nm=base.M3S_nm_edge.median(),
            median_wafer_edge_prediction_RMSE_nm=prediction.RMSE_nm.median(), median_wafer_edge_prediction_mean_nm=prediction.mean_nm.median()))
ep_product_summary = pd.DataFrame(ep_summary_rows)
display(ep_input_audit)
display(ep_product_summary)
display(ep_prediction_coverage)


## 06. 제품별 그래프


In [ ]:
ep_products = [str(s["product"]) for s in EP_SOURCES]
ep_product_labels = {p: f"{p} [{EP_PRODUCT_AXES[p][0].upper()}]" for p in ep_products}
ep_colors = dict(zip(ep_products, plt.get_cmap("tab10")(np.linspace(0, .9, len(ep_products)))))
ep_plot_paths = []


def ep_save_plot(fig, name):
    for extension in ["png", "pdf"]:
        path = EP_OUTPUT_DIR / "plots" / f"{name}.{extension}"
        fig.savefig(path, dpi=170, bbox_inches="tight")
        ep_plot_paths.append(path)
    display(fig)
    plt.close(fig)


def ep_boxplot(panel, data, column, title):
    rng = np.random.default_rng(20261029)
    for index, product in enumerate(ep_products):
        values = pd.to_numeric(data.loc[data["product"].eq(product), column], errors="coerce").dropna().to_numpy(float)
        if len(values):
            panel.boxplot([values], positions=[index], widths=.5, patch_artist=True, showfliers=False, boxprops=dict(facecolor=ep_colors[product], alpha=.3), medianprops=dict(color="black"))
            panel.scatter(index+rng.uniform(-.12, .12, len(values)), values, s=14, alpha=.65, color=ep_colors[product])
        else:
            panel.text(index, .5, "No eligible data", ha="center", rotation=90, transform=panel.get_xaxis_transform())
    labels = [f"{ep_product_labels[p]}\nN={data.loc[data['product'].eq(p), column].notna().sum()} wafers" for p in ep_products]
    panel.set_xticks(np.arange(len(ep_products)), labels)
    panel.set_xlim(-.6, len(ep_products)-.4)
    panel.set_title(title)
    panel.set_ylabel("nm")
    panel.grid(axis="y", alpha=.25)


for axis in ["critical"]:
    # 제품별 설정 방향을 같은 그림에 표시. Excel의 axis는 실제 x/y를 유지.
    base = ep_wafer_comparison
    prediction = ep_wafer_prediction
    fig, panels = plt.subplots(1, 2, figsize=(13, 5))
    ep_boxplot(panels[0], base, "M3S_nm_inner", "R <= 144 mm: |mean| + 3 sigma")
    ep_boxplot(panels[1], base, "M3S_nm_edge", "144 < R <= 147 mm: |mean| + 3 sigma")
    fig.suptitle(f"OCO {axis.upper()}: unscaled input after fixed K-MRC, before CPE | each dot = wafer")
    fig.tight_layout()
    ep_save_plot(fig, f"01_Input_M3S_{axis.upper()}")

    fig, panel = plt.subplots(figsize=(7, 6))
    for product in ep_products:
        part = base.loc[base["product"].eq(product)].dropna(subset=["M3S_nm_inner", "M3S_nm_edge"])
        panel.scatter(part.M3S_nm_inner, part.M3S_nm_edge, color=ep_colors[product], s=22, alpha=.7, label=f"{ep_product_labels[product]} (N={len(part)})")
    limits = base[["M3S_nm_inner", "M3S_nm_edge"]].to_numpy(float)
    finite = limits[np.isfinite(limits)]
    if len(finite):
        lo, hi = float(finite.min()), float(finite.max())
        panel.plot([lo, hi], [lo, hi], ":", color="gray", label="Equal M3S")
    panel.set(xlabel="Inner input M3S [nm]", ylabel="Edge input M3S [nm]", title=f"OCO {axis.upper()}: same-wafer input comparison (before CPE)")
    panel.legend(title="Product")
    panel.grid(alpha=.25)
    fig.tight_layout()
    ep_save_plot(fig, f"02_Edge_vs_inner_input_{axis.upper()}")

    fig, panels = plt.subplots(1, 2, figsize=(13, 5))
    ep_boxplot(panels[0], prediction, "RMSE_nm", "Edge - inner-only CPE prediction: RMSE")
    ep_boxplot(panels[1], prediction, "mean_nm", "Edge - inner-only CPE prediction: signed mean")
    panels[1].axhline(0, color="gray", linewidth=.8)
    fig.suptitle(f"OCO {axis.upper()}: eligible inner references only | each dot = wafer | check coverage")
    fig.tight_layout()
    ep_save_plot(fig, f"03_Edge_prediction_error_{axis.upper()}")

    diag = ep_reference_diagnostics
    fig, panels = plt.subplots(1, 2, figsize=(14, 5))
    for index, product in enumerate(ep_products):
        part = diag.loc[diag["product"].eq(product)]
        panels[0].bar(index, len(part), color="lightgray")
        panels[0].bar(index, int(part.eligible.sum()), color=ep_colors[product])
        panels[0].text(index, len(part), f"{int(part.eligible.sum())}/{len(part)}", ha="center", va="bottom")
        for eligible, marker in [(True, "o"), (False, "x")]:
            selected = part.loc[part.eligible.eq(eligible)].dropna(subset=["ridge_edge_noise_gain_max", "candidate_edge_RMSE_nm"])
            panels[1].scatter(selected.ridge_edge_noise_gain_max, selected.candidate_edge_RMSE_nm, color=ep_colors[product], marker=marker, s=23, alpha=.7, label=f"{ep_product_labels[product]}: {'eligible' if eligible else 'excluded'}")
    panels[0].set_xticks(np.arange(len(ep_products)), [ep_product_labels[p] for p in ep_products])
    panels[0].set(ylabel="Shot count", title="Eligible / all shots with edge points (gray = all)")
    panels[1].axvline(EP_MAX_EDGE_NOISE_GAIN, color="gray", linestyle=":")
    panels[1].set(xlabel="Max edge prediction noise gain (linear ridge proxy)", ylabel="Candidate edge prediction RMSE [nm]", title="Geometry / regularization diagnostic")
    panels[1].legend(fontsize=8, title="Product / reference screening")
    for panel in panels:
        panel.grid(alpha=.2)
    fig.tight_layout()
    ep_save_plot(fig, f"04_Reference_coverage_{axis.upper()}")

    fig, panel = plt.subplots(figsize=(10, 5))
    profile = ep_radial_profiles
    any_profile = False
    for product in ep_products:
        part = profile.loc[profile["product"].eq(product)]
        if part.empty:
            continue
        grouped = part.groupby("radius_mid_mm", observed=True).RMSE_nm
        median, lower, upper = grouped.median(), grouped.quantile(.25), grouped.quantile(.75)
        panel.plot(median.index, median.to_numpy(), ".-", color=ep_colors[product], label=ep_product_labels[product])
        panel.fill_between(median.index, lower.to_numpy(), upper.to_numpy(), alpha=.15, color=ep_colors[product])
        any_profile = True
    panel.axvline(EP_INNER_RADIUS_UM/1000, color="black", linestyle=":", label="144 mm boundary")
    if not any_profile:
        panel.text(.5, .5, "No eligible inner reference for radial comparison", ha="center", transform=panel.transAxes)
    panel.set(xlabel="Actual point radius [mm]", ylabel="Reference residual RMSE [nm]", title=f"OCO {axis.upper()}: wafer median / IQR | inner = fitted, edge = predicted")
    panel.legend(title="Product")
    panel.grid(alpha=.25)
    fig.tight_layout()
    ep_save_plot(fig, f"05_Radial_reference_error_{axis.upper()}")


## 07. Excel·전체 Parquet·실행 설정 저장


In [ ]:
ep_notes = pd.DataFrame([
    ["입력", "제품별 27_OCO_FINAL_POINTS 전체 Parquet(또는 전체 Excel) + 같은 실행의 extension_settings.json. ADI/sweep 파일 불필요."],
    ["제품별 방향", "; ".join(f"{product}: {axes[0].upper()}" for product, axes in EP_PRODUCT_AXES.items())],
    ["회귀 및 비교", "각 제품에서 지정한 방향만 통계·회귀·예측. X는 기존 X21, Y는 기존 Y17을 사용하며 회전하지 않음. CRITICAL 그래프는 제품별 해당 방향을 함께 표시하고 축/범례에 X/Y 명시."],
    ["분석 값", "input100 = 저장된 raw_minus_total_fit + (1-measurement_scale)*원래 X/Y 측정값. 기적용 MRC/PSM decorrection 및 고정 K-MRC는 유지. 최종 residual 자체를 역배율로 나누지 않음."],
    ["영역", "실제 point 반경 <=144mm: INNER. 144mm 초과~147mm 이하: EDGE. 반경 미상/범위 밖 제외 수는 Input_audit에 기록."],
    ["입력 통계", "CPE 제거 전 input100의 mean/3sigma/M3S/RMSE. 위치·실제 변형 차이가 섞이므로 edge 측정 noise만의 크기로 해석하지 않음."],
    ["참조 모델", "같은 shot의 INNER 유효 point만 사용해 기존 CPE38(X21/Y17) 회귀. EDGE 값은 회귀 입력·ridge/span 선택에 사용하지 않음. 좌표와 shot metadata는 평가 grid에 사용."],
    ["설정", "제품별 28번 ridge/G-opt/span/virtual 설정 복원. 가상 point의 radius clip은 저장된 설정 그대로. 실제 EDGE point의 예측 noise gain은 별도 계산."],
    ["유효 point", "선택한 방향 값과 X/Y 좌표가 유효한 point만 참조 회귀에 사용. 분석하지 않는 방향의 결측은 선택 방향 분석에 영향을 주지 않음."],
    ["내측 최소 point", f"X 21 + {EP_MIN_EXTRA_INNER_POINTS}, Y 17 + {EP_MIN_EXTRA_INNER_POINTS}; para 수 축소 없음. 부족/미식별 shot은 예측하지 않음."],
    ["참조 screening", f"안쪽 scaled design full rank, condition <= {EP_MAX_SCALED_CONDITION:g}, edge ridge noise gain <= {EP_MAX_EDGE_NOISE_GAIN:g}, 유한 예측·G-opt·span 확인. 이 임계값은 연구용 진단 설정이며 공정 spec이 아님."],
    ["noise gain", "독립·동일 측정 noise 가정의 선형 ridge 전달 행렬 행 norm. Box 제약이 작동하면 근사값. 낮아도 모델 편향·물리적 외삽 오차가 없음을 보장하지 않음."],
    ["예측 오차", "edge_deviation = EDGE input100 - INNER-only CPE prediction. reference eligible shot만 Wafer/Shot_prediction 및 제품 요약에 포함. 부호 있는 mean은 편향, RMSE는 편향과 산포를 함께 표현."],
    ["후보 예측", "불안정하지만 full rank인 참조의 후보 값은 Shot_reference.candidate_edge_RMSE_nm와 point의 inner_prediction/reference_residual에 보존. 주요 예측 통계에서는 제외."],
    ["coverage", "각 wafer/제품에서 예측 가능한 EDGE point / 전체 EDGE point. 커버리지와 shot 종류가 다른 제품의 RMSE를 단독 순위로 해석하지 않음."],
    ["빈 결과", "안쪽만/edge만 있는 shot, 안쪽 point 부족, 미식별 또는 불안정 참조는 상태와 개수 기록. 예측 불가를 오차 0으로 대체하지 않음."],
    ["측정 noise와 구분", "예측 오차에는 실제 edge 고유 패턴, 모델 불일치, 측정 noise, 외삽 오차가 함께 포함. 반복 계측 없이 이 성분들을 분리하지 않음."],
    ["통계", f"M3S=abs(mean)+3*std, ddof={EP_STD_DDOF}. Point 수 가중 wafer 통계이며 shot 평균의 평균이 아님. Product_summary는 wafer 통계의 중앙값."],
    ["단위", "통계·그래프 nm. Point의 input100/prediction/deviation 및 기존 fitting 값은 µm. 좌표·반경 µm."],
    ["그래프", "각 점은 wafer. 제품 색은 범례/축 이름에 표시. 03은 eligible prediction만 표시. 04의 회색 막대는 전체 edge shot, 유색 막대는 eligible shot."],
    ["반경 profile", "Eligible edge shot만 포함. 안쪽 residual은 in-sample, edge는 out-of-sample이므로 양쪽 차이를 순수 noise 차이로 해석하지 않음. Band는 wafer별 값의 25~75%, 신뢰구간이 아님."],
    ["원인 판정", "이번 분석은 edge-안쪽 차이의 관측 자료를 제공하며 최적 배율이나 원인 확정을 자동 판정하지 않음."],
], columns=["item", "description"])
ep_tables = {"Product_summary": ep_product_summary, "Wafer_input_comparison": ep_wafer_comparison, "Wafer_input_stats": ep_wafer_input,
             "Mixed_shot_input_comparison": ep_shot_comparison, "Shot_input_stats": ep_shot_input, "Wafer_prediction": ep_wafer_prediction, "Shot_prediction": ep_shot_prediction,
             "Shot_reference": ep_reference_diagnostics, "Reference_coverage": ep_prediction_coverage, "Reference_coefficients": ep_reference_coefficients,
             "Radial_profile": ep_radial_profiles, "Input_audit": ep_input_audit, "Input_files": pd.DataFrame(ep_input_inventory), "Notes": ep_notes}


def ep_excel_safe(frame):
    out = frame.copy()
    for col in out:
        if col.endswith("transn_seq") or col in ["apc_hist_index_no", "apc_trocs_hist_index_no"]:
            out[col] = out[col].map(lambda value: str(int(value)) if pd.notna(value) else "")
    return out


ep_excel_path = EP_OUTPUT_DIR / "OCO_Edge_Inner_Comparison.xlsx"
with pd.ExcelWriter(ep_excel_path, engine="openpyxl") as writer:
    for name, table in ep_tables.items():
        table.to_parquet(EP_OUTPUT_DIR / "tables" / f"{name}.parquet", index=False)
        export = ep_excel_safe(table)
        for offset in range(0, max(1, len(export)), 1048575):
            suffix = "" if len(export) <= 1048575 else f"_{offset//1048575+1}"
            export.iloc[offset:offset+1048575].to_excel(writer, sheet_name=name[:31-len(suffix)]+suffix, index=False)
ep_configuration = dict(sources=ep_run_sources, product_axes=EP_PRODUCT_AXES, axes=EP_AXES, inner_radius_um=EP_INNER_RADIUS_UM, max_radius_um=EP_MAX_RADIUS_UM, std_ddof=EP_STD_DDOF,
    minimum_extra_inner_points=EP_MIN_EXTRA_INNER_POINTS, max_scaled_condition=EP_MAX_SCALED_CONDITION, max_edge_noise_gain=EP_MAX_EDGE_NOISE_GAIN,
    svd_rcond=EP_SVD_RCOND, radial_bins_mm=EP_RADIAL_BINS_MM.tolist(), point_unit="um", statistics_unit="nm", fixed_kmrc=True, edge_values_used_for_reference_fit=False)
(EP_OUTPUT_DIR / "comparison_settings.json").write_text(json.dumps(ep_configuration, ensure_ascii=False, indent=2), encoding="utf-8")
display(FileLink(os.path.relpath(ep_excel_path, Path.cwd()), result_html_prefix="제품별 edge/inner 비교 Excel: "))
for path in ep_plot_paths:
    if path.suffix == ".png":
        display(FileLink(os.path.relpath(path, Path.cwd()), result_html_prefix="그래프: "))
print("분석 완료:", EP_OUTPUT_DIR)
